In [ ]:

#!/usr/bin/env python3
"""
Agentic AI Telegram Bot (v20+)
--------------------------------
This bot accepts voice and text messages from users. For voice messages,
the bot downloads and transcribes the voice file (using a simulated transcription).
The processed text is then passed to an AI Agent that uses:
  • Chat Model: ChatGPT-4 via OpenAI’s API,
  • Memory: In‑memory Windows buffer and MongoDB,
  • Tools: A set of simulated external APIs (RAG retrieval, meeting setter,
    financial news, price data, chart image, content creator).

Dependencies:
   pip install python-telegram-bot pymongo openai

Configuration:
   Set the environment variables:
      TELEGRAM_BOT_TOKEN, OPENAI_API_KEY, MONGODB_URI
"""

import os
import logging
from datetime import datetime

# Telegram bot libraries for v20+
from telegram import Update
from telegram.ext import (
    Application,
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters,
)

import openai  # OpenAI API client

# For MongoDB – ensure the connection string is properly set via environment variable.
from pymongo import MongoClient

########################################################################
# CONFIGURATION (Set these as environment variables or hardcode for testing)
########################################################################
TELEGRAM_BOT_TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN", "YOUR_TELEGRAM_BOT_TOKEN_HERE")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY_HERE")
MONGODB_URI = os.environ.get("MONGODB_URI", "mongodb://localhost:27017")

openai.api_key = OPENAI_API_KEY

# Enable Logging
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", level=logging.INFO
)
logger = logging.getLogger(__name__)

########################################################################
# MEMORY MANAGER: In‑Memory and MongoDB
########################################################################
class MemoryManager:
    def __init__(self):
        # Windows buffer (in‑memory conversation log)
        self.buffer = []
        # Set up MongoDB connection
        self.client = MongoClient(MONGODB_URI)
        self.db = self.client["agentic_ai"]
        self.collection = self.db["conversations"]

    def add_message(self, role: str, message: str):
        """Store a message in both in‑memory buffer and MongoDB."""
        timestamp = datetime.utcnow().isoformat()
        entry = {"role": role, "message": message, "timestamp": timestamp}
        self.buffer.append(entry)
        # Persist to MongoDB for record
        self.collection.insert_one(entry)

    def get_memory(self):
        """Return the in‑memory conversation log."""
        return self.buffer

########################################################################
# CHAT MODEL: Wrapper for ChatGPT-4 API
########################################################################
class ChatModel:
    def __init__(self, model_name="gpt-4"):
        self.model_name = model_name

    def chat(self, conversation: list) -> str:
        """
        Given a conversation (list of dicts with 'role' and 'message'),
        call ChatGPT-4 API to generate a response.
        """
        try:
            messages = [{"role": m["role"], "content": m["message"]} for m in conversation]
            response = openai.ChatCompletion.create(
                model=self.model_name,
                messages=messages,
                temperature=0.7,
                max_tokens=150,
            )
            answer = response.choices[0].message["content"].strip()
            return answer
        except Exception as e:
            logger.error("Error calling ChatGPT API: %s", e)
            return "I'm sorry, I'm having trouble generating a response right now."

########################################################################
# TOOL MANAGER: Simulated External API Calls
########################################################################
class ToolManager:
    def consult_meeting_setter(self, input_text: str) -> str:
        return "Consulting Meeting Setter API Response: Your meeting is scheduled."

    def get_financial_news(self, input_text: str) -> str:
        return "Financial News API Response: Latest market news retrieved."

    def get_price_data(self, input_text: str) -> str:
        return "Price Data API Response: The latest price data is $123.45."

    def get_chart_image(self, input_text: str) -> str:
        return "Chart Image API Response: https://example.com/chart.png"

    def create_content(self, input_text: str) -> str:
        return "Content Creator API Response: Here is a creative piece based on your prompt."

    def rag_retrieval(self, input_text: str) -> str:
        return "RAG Retrieval API Response: Relevant documents have been retrieved."

    def decide_and_invoke_tool(self, input_text: str) -> str:
        """
        Decides which tool(s) to invoke based on keywords in the input.
        """
        tool_outputs = []
        lower_input = input_text.lower()

        if "meeting" in lower_input or "appointment" in lower_input:
            tool_outputs.append(self.consult_meeting_setter(input_text))
        if "news" in lower_input or "tweet" in lower_input:
            tool_outputs.append(self.get_financial_news(input_text))
        if "price" in lower_input or "data" in lower_input:
            tool_outputs.append(self.get_price_data(input_text))
        if "chart" in lower_input:
            tool_outputs.append(self.get_chart_image(input_text))
        if "create" in lower_input or "content" in lower_input:
            tool_outputs.append(self.create_content(input_text))
        if "document" in lower_input or "knowledge" in lower_input:
            tool_outputs.append(self.rag_retrieval(input_text))

        if tool_outputs:
            return "\n".join(tool_outputs)
        else:
            return ""

########################################################################
# AI AGENT: Combining Memory, Chat Model, and Tools
########################################################################
class AIAgent:
    def __init__(self):
        self.memory = MemoryManager()
        self.chat_model = ChatModel()
        self.tools = ToolManager()

    def process_input(self, input_text: str) -> str:
        """
        Process user input by:
          1. Logging input.
          2. Deciding on external tool invocation.
          3. Preparing conversation context.
          4. Generating a response.
          5. Logging the response.
        """
        logger.info("Processing input: %s", input_text)
        self.memory.add_message("user", input_text)

        tool_info = self.tools.decide_and_invoke_tool(input_text)
        tool_text = ("\n[Tool Output]\n" + tool_info) if tool_info else ""

        system_prompt = ("You are a versatile agent that can assist with multiple tasks "
                         "using various external tools when needed.")
        conversation = [{"role": "system", "message": system_prompt}]

        for entry in self.memory.get_memory():
            conversation.append({"role": entry["role"], "message": entry["message"]})

        full_input = input_text + tool_text
        conversation.append({"role": "user", "message": full_input})

        response = self.chat_model.chat(conversation)
        self.memory.add_message("assistant", response)
        return response

########################################################################
# UTILITY FUNCTION FOR VOICE TRANSCRIPTION (Simulated)
########################################################################
def voice_to_text(file_path: str) -> str:
    """
    Dummy transcription function.
    Replace with actual transcription API as needed.
    """
    logger.info("Transcribing voice file %s", file_path)
    return "Simulated transcription of voice file."

########################################################################
# TELEGRAM BOT HANDLERS (ASYNC)
########################################################################
# Global AI agent instance.
agent_instance = AIAgent()

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Send a welcome message when /start command is issued."""
    await update.message.reply_text("Hello! I am your agentic AI bot. Send a voice or text message.")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Handle incoming voice or text messages."""
    input_text = ""
    message = update.message

    if message.voice:
        file = await message.voice.get_file()
        file_path = f"voice_{message.voice.file_id}.ogg"
        await file.download(custom_path=file_path)
        logger.info("Downloaded voice file to %s", file_path)
        input_text = voice_to_text(file_path)
        os.remove(file_path)
    elif message.text:
        input_text = message.text
    else:
        await message.reply_text("Sorry, I only process text and voice messages.")
        return

    logger.info("Incoming user text: %s", input_text)
    response = agent_instance.process_input(input_text)
    logger.info("Agent Response: %s", response)
    await message.reply_text(response)

########################################################################
# MAIN FUNCTION: SET UP AND RUN THE TELEGRAM BOT
########################################################################
def main() -> None:
    """Start the bot using ApplicationBuilder."""
    application = ApplicationBuilder().token(TELEGRAM_BOT_TOKEN).build()

    application.add_handler(CommandHandler("start", start))
    application.add_handler(MessageHandler(filters.VOICE | filters.TEXT, handle_message))

    logger.info("Bot started. Listening for messages...")
    application.run_polling()

if __name__ == "__main__":
    main()

────────────────────────────────────────────────────────────
README Summary
────────────────────────────────────────────────────────────
Agentic AI Telegram Bot (v20+)
-----------------------------
1. This script uses python‑telegram‑bot v20+ with asynchronous handlers.
2. It processes both text and voice messages:
   • Voice messages are downloaded and “transcribed” (using a dummy transcription).
   • Text messages are processed directly.
3. The AI Agent consists of a Memory Manager (Windows buffer + MongoDB),
   a Chat Model (ChatGPT‑4 via OpenAI API), and a Tool Manager
   invoking simulated external APIs.
4. Setup:
   • Set environment variables: TELEGRAM_BOT_TOKEN, OPENAI_API_KEY, and MONGODB_URI.
   • Install dependencies: pip install python-telegram-bot pymongo openai
   • Run the script: python agentic_telegram_bot_v20.py

This updated code is now in line with the latest version of python-telegram-bot.